In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
pd.set_option('display.float_format', '{:.2f}'.format)

import psycopg2
import matplotlib.pyplot as plt 
import seaborn as sns
from dotenv import load_dotenv
import os

from scipy import stats

from config.settings import DB_CONFIG

RANDOM_STATE = 42

conn = psycopg2.connect(**DB_CONFIG)

query_silver = 'select * from silver_listings where is_primary'
query_listings_price_history = 'select * from listings join price_history using(cian_id)'

query_bm = 'select * from gold_building_monthly'
query_seg = 'select * from gold_segment_index'
query_ret = 'select * from gold_building_returns'
query_dist = 'select * from gold_district_stats'
query_ml = 'select * from gold_ml_features'
query_listings = 'select * from listings join price_history using(cian_id)'
query_silver = 'select * from silver_listings where is_primary'

In [ ]:
# lph = pd.read_sql(query_listings_price_history, conn)
dfs = pd.read_sql(query_silver, conn)

display(dfs.shape)
display(dfs.info())
display(dfs.head(2))


(229971, 69)

<class 'pandas.DataFrame'>
RangeIndex: 229971 entries, 0 to 229970
Data columns (total 69 columns):
 #   Column               Non-Null Count   Dtype              
---  ------               --------------   -----              
 0   listing_id           229971 non-null  int64              
 1   cian_id              80339 non-null   float64            
 2   source               229971 non-null  str                
 3   url                  229971 non-null  str                
 4   dedup_group_id       229971 non-null  int64              
 5   is_primary           229971 non-null  bool               
 6   group_size           229971 non-null  int64              
 7   group_min_price      229971 non-null  int64              
 8   group_max_price      229971 non-null  int64              
 9   group_seller_types   65299 non-null   str                
 10  price                229971 non-null  int64              
 11  price_per_m2         229971 non-null  int64              
 12  city         

None

,listing_id,cian_id,source,url,dedup_group_id,is_primary,group_size,group_min_price,group_max_price,group_seller_types,...,seller_type,is_active,first_seen_at,last_seen_at,has_coords,has_year_built,has_pub_date,data_quality_score,etl_loaded_at,etl_version
0,2959209,NaN,kaggle_hishamhaydar,kaggle_hishamhaydar_8593,1677515063,True,1,10626000,10626000,NaN,...,NaN,None,NaT,NaT,False,False,False,4,2026-04-08 07:47:02.816988+00:00,1.0
1,2959217,NaN,kaggle_hishamhaydar,kaggle_hishamhaydar_8601,1752692431,True,1,10185120,10185120,NaN,...,NaN,None,NaT,NaT,False,False,False,4,2026-04-08 07:47:02.816988+00:00,1.0


In [13]:
dfs.loc[dfs['region'] == 'Москва', 'price_per_m2'].median()

np.float64(459236.0)

In [14]:
dfs.pivot_table(index='region', columns='is_new_building', values='price_per_m2', aggfunc='median')

is_new_building,False,True
region,,
ВАО,291376.00,406024.00
ЗАО,462312.00,478210.00
Москва,427958.00,522467.00
Московская область,179133.00,229430.00
САО,365297.00,446127.50
СВАО,317401.50,448000.00
СЗАО,419822.50,489780.00
ЦАО,766633.00,895402.00
ЮАО,322800.00,473684.00


In [ ]:
# silver = pd.read_sql(query_silver, conn)

'''
temp_1 = (lph.groupby('cian_id')['price'].max() - lph.groupby('cian_id')['price'].min())
temp_1['cian_id'] = temp_1.index
temp_1 = temp_1.reset_index(drop=True)
temp_1 = temp_1.iloc[:, 1:]
temp_1.columns = ['price_diff', 'cian_id']
temp_1 = temp_1[(temp_1['price_diff'] > 10_000) & (temp_1['price_diff'] < 100_000_000)].sort_values('price_diff', ascending=False)

temp_2 = pd.DataFrame((lph.groupby('cian_id')['recorded_at'].max() - lph.groupby('cian_id')['recorded_at'].min()))
temp_2['cian_id'] = temp_2.index
temp_2 = temp_2.reset_index(drop=True)
temp_2.columns = ['date_diff', 'cian_id']
temp_2 = temp_2[temp_2['date_diff'] > '1 day'].sort_values('date_diff', ascending=False)

temp = pd.merge(temp_1, temp_2, how='left', on='cian_id')
'''